In [1]:
"""
Telecom Customer Churn Data Synthesizer
Generates ~500k rows of realistic Indian telecom customer data
with statistically plausible correlations baked in.

Usage:
    pip install numpy pandas scipy faker tqdm
    python generate_telecom_data.py
"""

import numpy as np
import pandas as pd
import random
import sys
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
random.seed(42)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
N = 500_000
OUTPUT_FILE = "telecom_churn_data.csv"

# ─────────────────────────────────────────────
# LOOKUP TABLES  (realistic Indian telecom context)
# ─────────────────────────────────────────────

REGIONS = {
    "Mumbai":       0.12,
    "Delhi":        0.11,
    "Bangalore":    0.10,
    "Hyderabad":    0.08,
    "Chennai":      0.07,
    "Kolkata":      0.07,
    "Pune":         0.06,
    "Ahmedabad":    0.05,
    "Jaipur":       0.05,
    "Lucknow":      0.05,
    "Bhopal":       0.04,
    "Patna":        0.04,
    "Tier3_North":  0.08,
    "Tier3_South":  0.08,
}

PLAN_TIERS = ["Prepaid_Basic", "Prepaid_Mid", "Prepaid_Premium",
              "Postpaid_Basic", "Postpaid_Premium", "Postpaid_Family"]

# (bundle_name, monthly_cost_range, ott_platforms, churn_stickiness)
BUNDLES = {
    "No_Bundle":           (149,  299,  [],                                   0.00),
    "Basic_OTT":           (299,  399,  ["Hotstar"],                          0.10),
    "Entertainment":       (399,  599,  ["Hotstar", "SonyLIV"],               0.18),
    "Premium_OTT":         (599,  799,  ["Netflix", "Hotstar", "Prime"],      0.28),
    "Gaming_Pack":         (349,  499,  ["GameStream"],                       0.12),
    "Family_Bundle":       (799, 1199,  ["Hotstar", "Prime", "SonyLIV"],      0.30),
    "Business_Bundle":     (999, 1999,  ["Microsoft365", "ZoomPro"],          0.25),
    "Student_Pack":        (199,  299,  ["Hotstar"],                          0.08),
    "Fiber_OTT_Combo":     (699, 1299,  ["Netflix", "Prime", "Hotstar"],      0.35),
    "International_Roam":  (999, 2999,  [],                                   0.20),
}

CHURN_TYPES = ["Stayed", "Bundle_Switch", "Company_Churn"]

NETWORK_TYPES = ["4G", "5G", "4G/5G_Dual"]

# ─────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────

def weighted_choice(d: dict):
    keys = list(d.keys())
    weights = list(d.values())
    return np.random.choice(keys, p=np.array(weights) / sum(weights))


def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


# ─────────────────────────────────────────────
# CORE GENERATOR
# ─────────────────────────────────────────────

def generate_customer(cid: int) -> dict:
    # ── Demographics ──────────────────────────────────────────────────────────
    region         = weighted_choice(REGIONS)
    is_metro       = region not in ["Tier3_North", "Tier3_South"]
    age            = int(np.random.normal(35, 12))
    age            = clamp(age, 18, 75)
    is_student     = (18 <= age <= 24)
    is_senior      = (age >= 55)
    gender         = np.random.choice(["Male", "Female", "Other"], p=[0.54, 0.44, 0.02])
    occupation     = np.random.choice(
        ["Salaried", "Self_Employed", "Student", "Homemaker", "Retired"],
        p=[0.45, 0.20, 0.15, 0.12, 0.08]
    )

    # ── Plan ─────────────────────────────────────────────────────────────────
    if is_student:
        plan_tier = np.random.choice(["Prepaid_Basic", "Prepaid_Mid", "Student_Pack"],
                                     p=[0.5, 0.35, 0.15])
    elif occupation == "Salaried" and is_metro:
        plan_tier = np.random.choice(PLAN_TIERS, p=[0.10, 0.20, 0.20, 0.15, 0.25, 0.10])
    elif occupation == "Retired":
        plan_tier = np.random.choice(PLAN_TIERS, p=[0.40, 0.30, 0.10, 0.10, 0.05, 0.05])
    else:
        plan_tier  = np.random.choice(PLAN_TIERS)

    is_postpaid    = plan_tier.startswith("Postpaid")
    is_prepaid     = not is_postpaid

    # ── Tenure ───────────────────────────────────────────────────────────────
    # Bimodal: lots of new + loyal customers
    if np.random.rand() < 0.35:
        tenure_months = int(np.random.exponential(6))           # new customers
    else:
        tenure_months = int(np.random.normal(42, 20))           # loyal customers
    tenure_months = clamp(tenure_months, 1, 120)

    # ── Contract ─────────────────────────────────────────────────────────────
    if is_prepaid:
        contract_type = np.random.choice(
            ["Monthly", "Quarterly", "Annual"],
            p=[0.60, 0.25, 0.15]
        )
    else:
        contract_type = np.random.choice(
            ["Monthly", "Annual", "2_Year"],
            p=[0.30, 0.45, 0.25]
        )
    contract_length_months = {"Monthly": 1, "Quarterly": 3, "Annual": 12, "2_Year": 24}[contract_type]

    # ── Bundle ───────────────────────────────────────────────────────────────
    bundle_weights = {
        "No_Bundle":          0.25,
        "Basic_OTT":          0.15,
        "Entertainment":      0.15,
        "Premium_OTT":        0.10,
        "Gaming_Pack":        0.08,
        "Family_Bundle":      0.08,
        "Business_Bundle":    0.06,
        "Student_Pack":       0.04,
        "Fiber_OTT_Combo":    0.05,
        "International_Roam": 0.04,
    }
    # Adjust for demographics
    if is_student:
        bundle_weights["Student_Pack"] += 0.15
        bundle_weights["No_Bundle"]    -= 0.10
    if occupation == "Business" or plan_tier == "Postpaid_Premium":
        bundle_weights["Business_Bundle"]   += 0.10
        bundle_weights["International_Roam"] += 0.05
    if occupation == "Salaried" and is_metro:
        bundle_weights["Premium_OTT"]  += 0.05
        bundle_weights["Family_Bundle"] += 0.05

    bundle_name = weighted_choice(bundle_weights)
    b_min, b_max, ott_list, bundle_stickiness = BUNDLES[bundle_name]
    monthly_bundle_cost = round(np.random.uniform(b_min, b_max), 2)
    ott_platforms       = "|".join(ott_list) if ott_list else "None"
    num_ott_platforms   = len(ott_list)

    bundle_switches_6mo = 0
    if tenure_months > 3:
        switch_prob = 0.05 if bundle_stickiness > 0.2 else 0.15
        bundle_switches_6mo = int(np.random.poisson(switch_prob * 3))
        bundle_switches_6mo = clamp(bundle_switches_6mo, 0, 5)

    # ── Usage: Data ──────────────────────────────────────────────────────────
    if plan_tier in ["Prepaid_Basic", "Student_Pack"]:
        daily_data_limit_gb  = np.random.choice([0.5, 1.0, 1.5, 2.0], p=[0.3, 0.4, 0.2, 0.1])
    elif plan_tier in ["Prepaid_Mid", "Postpaid_Basic"]:
        daily_data_limit_gb  = np.random.choice([1.5, 2.0, 3.0], p=[0.3, 0.5, 0.2])
    else:
        daily_data_limit_gb  = np.random.choice([2.0, 3.0, "Unlimited"], p=[0.2, 0.3, 0.5])
    daily_data_limit_gb  = 100.0 if daily_data_limit_gb == "Unlimited" else float(daily_data_limit_gb)

    avg_daily_data_used_gb = round(
        clamp(np.random.normal(daily_data_limit_gb * 0.72, daily_data_limit_gb * 0.25), 0.1, daily_data_limit_gb * 1.1), 2
    )
    data_usage_ratio       = round(avg_daily_data_used_gb / daily_data_limit_gb, 3)
    total_data_used_gb     = round(avg_daily_data_used_gb * 30, 1)
    data_overuse_days_mo   = int(clamp(np.random.poisson(max(0, (data_usage_ratio - 0.85) * 15)), 0, 30))

    # ── Usage: Calls & SMS ───────────────────────────────────────────────────
    monthly_call_minutes = int(clamp(np.random.normal(320, 140), 10, 1200))
    monthly_sms_count    = int(clamp(np.random.poisson(45), 0, 400))
    roaming_calls_mo     = int(np.random.poisson(2.5)) if is_metro else int(np.random.poisson(0.8))
    international_calls  = int(np.random.poisson(1.2)) if occupation in ["Salaried", "Self_Employed"] else 0

    # ── Financial ────────────────────────────────────────────────────────────
    base_plan_cost = {
        "Prepaid_Basic":    149,
        "Prepaid_Mid":      249,
        "Prepaid_Premium":  399,
        "Postpaid_Basic":   499,
        "Postpaid_Premium": 999,
        "Postpaid_Family":  1499,
        "Student_Pack":     199,
    }.get(plan_tier, 299)

    monthly_arpu      = round(base_plan_cost + monthly_bundle_cost * 0.6 +
                              (roaming_calls_mo * 2.5) + (international_calls * 15) +
                              np.random.normal(0, 30), 2)
    monthly_arpu      = max(99.0, monthly_arpu)
    total_revenue_ltv = round(monthly_arpu * tenure_months, 2)

    payment_delays_6mo = 0
    if is_prepaid:
        payment_delays_6mo = int(np.random.poisson(0.8))
    else:
        payment_delays_6mo = int(np.random.poisson(0.3))
    payment_delays_6mo = clamp(payment_delays_6mo, 0, 6)

    recharge_frequency = "N/A"
    days_since_last_recharge = 0
    if is_prepaid:
        recharge_frequency = np.random.choice(
            ["Weekly", "Bi-Weekly", "Monthly", "Irregular"],
            p=[0.20, 0.30, 0.35, 0.15]
        )
        days_since_last_recharge = int(clamp(np.random.exponential(12), 1, 60))

    # ── Network & Support ────────────────────────────────────────────────────
    network_type        = np.random.choice(NETWORK_TYPES, p=[0.45, 0.35, 0.20])
    avg_network_score   = round(clamp(np.random.normal(3.6, 0.8), 1.0, 5.0), 1)
    outage_incidents_mo = int(np.random.poisson(0.4 if avg_network_score > 3 else 1.5))
    support_tickets_6mo = int(np.random.poisson(
        0.5 + (1.5 if outage_incidents_mo > 1 else 0) + (1.0 if data_overuse_days_mo > 5 else 0)
    ))
    support_tickets_6mo = clamp(support_tickets_6mo, 0, 15)
    complaint_resolved  = np.random.choice(["Yes", "No", "Partial", "N/A"],
                                           p=[0.55, 0.15, 0.15, 0.15]) if support_tickets_6mo > 0 else "N/A"

    # ── Behavioral ───────────────────────────────────────────────────────────
    app_logins_per_week    = int(clamp(np.random.poisson(4.5), 0, 30))
    _hour_p = np.array([0.01]*6 + [0.03,0.04,0.05,0.06,0.06,0.05,
                                    0.05,0.05,0.05,0.05,0.05,0.06,
                                    0.07,0.07,0.06,0.05,0.04,0.02])
    _hour_p /= _hour_p.sum()
    peak_usage_hour        = int(np.random.choice(range(24), p=_hour_p))
    upgrade_offers_shown   = int(np.random.poisson(2.5))
    upgrade_offers_accepted= int(np.random.binomial(upgrade_offers_shown, 0.25))
    competitor_sim_owned   = np.random.choice([0, 1], p=[0.62, 0.38])

    # ── Retention History ────────────────────────────────────────────────────
    retention_offers_6mo   = int(np.random.poisson(0.8))
    retention_offer_accepted = (
        np.random.choice([1, 0], p=[0.55, 0.45]) if retention_offers_6mo > 0 else 0
    )
    loyalty_points_balance = int(clamp(np.random.exponential(800) * (tenure_months / 12), 0, 50000))

    # ─────────────────────────────────────────────────────────────────────────
    # CHURN PROBABILITY MODEL (logistic score)
    # Each factor adds/subtracts from a base log-odds
    # ─────────────────────────────────────────────────────────────────────────
    log_odds = -1.5   # base ~18% churn rate

    # Contract type (strong signal)
    if contract_type == "Monthly":          log_odds += 0.9
    elif contract_type == "Quarterly":      log_odds += 0.3
    elif contract_type == "Annual":         log_odds -= 0.4
    elif contract_type == "2_Year":         log_odds -= 0.8

    # Bundle stickiness
    log_odds -= bundle_stickiness * 2.5

    # Tenure: loyal customers churn less
    if tenure_months < 6:                   log_odds += 0.7
    elif tenure_months > 36:               log_odds -= 0.6

    # Data usage: near cap → frustrated → churn more
    if data_usage_ratio > 0.95:            log_odds += 0.5
    elif data_usage_ratio < 0.5:           log_odds += 0.3  # not using it → may leave

    # Support issues
    log_odds += support_tickets_6mo * 0.18
    if complaint_resolved == "No":         log_odds += 0.6
    elif complaint_resolved == "Yes":      log_odds -= 0.3

    # Network quality
    if avg_network_score < 2.5:            log_odds += 0.8
    elif avg_network_score > 4.2:          log_odds -= 0.4
    log_odds += outage_incidents_mo * 0.25

    # Payment delays
    log_odds += payment_delays_6mo * 0.25

    # Competitor SIM
    if competitor_sim_owned:               log_odds += 0.55

    # Bundle switches (frequent switchers are engaged but volatile)
    log_odds += bundle_switches_6mo * 0.15

    # Retention offers
    if retention_offer_accepted:           log_odds -= 0.5

    # App engagement
    if app_logins_per_week < 1:            log_odds += 0.4

    # Upgrade acceptance (engaged customer)
    if upgrade_offers_accepted > 0:        log_odds -= 0.3

    # Recharge irregularity (prepaid)
    if is_prepaid and recharge_frequency == "Irregular":
        log_odds += 0.5
    if days_since_last_recharge > 30:      log_odds += 0.4

    # Add noise
    log_odds += np.random.normal(0, 0.4)

    churn_prob = sigmoid(log_odds)

    # ── Assign Churn Type ─────────────────────────────────────────────────────
    rand = np.random.rand()
    if rand < (churn_prob * 0.55):
        churn_type = "Company_Churn"
    elif rand < (churn_prob * 0.55 + 0.12):   # ~12% of all customers switch bundles
        churn_type = "Bundle_Switch"
    else:
        churn_type = "Stayed"

    churn_binary = 1 if churn_type == "Company_Churn" else 0

    # ── Build record ──────────────────────────────────────────────────────────
    return {
        # IDs
        "customer_id":              f"CUST{cid:07d}",
        # Demographics
        "region":                   region,
        "age":                      age,
        "gender":                   gender,
        "occupation":               occupation,
        "is_metro":                 int(is_metro),
        # Plan
        "plan_tier":                plan_tier,
        "contract_type":            contract_type,
        "contract_length_months":   contract_length_months,
        "tenure_months":            tenure_months,
        "network_type":             network_type,
        # Bundle
        "bundle_name":              bundle_name,
        "ott_platforms":            ott_platforms,
        "num_ott_platforms":        num_ott_platforms,
        "monthly_bundle_cost":      monthly_bundle_cost,
        "bundle_switches_6mo":      bundle_switches_6mo,
        # Data usage
        "daily_data_limit_gb":      daily_data_limit_gb,
        "avg_daily_data_used_gb":   avg_daily_data_used_gb,
        "data_usage_ratio":         data_usage_ratio,
        "total_data_used_gb_mo":    total_data_used_gb,
        "data_overuse_days_mo":     data_overuse_days_mo,
        # Calls & SMS
        "monthly_call_minutes":     monthly_call_minutes,
        "monthly_sms_count":        monthly_sms_count,
        "roaming_calls_mo":         roaming_calls_mo,
        "international_calls_mo":   international_calls,
        # Financial
        "monthly_arpu":             monthly_arpu,
        "total_revenue_ltv":        total_revenue_ltv,
        "payment_delays_6mo":       payment_delays_6mo,
        "recharge_frequency":       recharge_frequency,
        "days_since_last_recharge": days_since_last_recharge,
        # Network & Support
        "avg_network_score":        avg_network_score,
        "outage_incidents_mo":      outage_incidents_mo,
        "support_tickets_6mo":      support_tickets_6mo,
        "complaint_resolved":       complaint_resolved,
        # Behavioral
        "app_logins_per_week":      app_logins_per_week,
        "peak_usage_hour":          peak_usage_hour,
        "competitor_sim_owned":     competitor_sim_owned,
        "upgrade_offers_shown":     upgrade_offers_shown,
        "upgrade_offers_accepted":  upgrade_offers_accepted,
        # Retention
        "retention_offers_6mo":     retention_offers_6mo,
        "retention_offer_accepted": retention_offer_accepted,
        "loyalty_points_balance":   loyalty_points_balance,
        # Targets
        "churn_probability_score":  round(churn_prob, 4),
        "churn_type":               churn_type,
        "churn":                    churn_binary,
    }


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

if __name__ == "__main__":
    print(f"Generating {N:,} customer records...")
    records = []
    for i in range(N):
        records.append(generate_customer(i + 1))
        if (i + 1) % 50_000 == 0:
            print(f"  {i+1:,} / {N:,} done...")

    print("Building DataFrame...")
    df = pd.DataFrame(records)

    print("\n── Dataset Overview ──────────────────────")
    print(f"Shape            : {df.shape}")
    print(f"File size (est.) : ~{df.memory_usage(deep=True).sum() / 1e6:.1f} MB in memory")
    print(f"\nChurn distribution:")
    print(df["churn_type"].value_counts(normalize=True).round(3).to_string())
    print(f"\nOverall churn rate : {df['churn'].mean():.2%}")
    print(f"Avg ARPU           : ₹{df['monthly_arpu'].mean():.2f}")
    print(f"Avg tenure         : {df['tenure_months'].mean():.1f} months")
    print(f"\nTop bundles:")
    print(df["bundle_name"].value_counts().head(5).to_string())

    print(f"\nSaving to {OUTPUT_FILE}...")
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"Done. File saved: {OUTPUT_FILE}")
    print(f"Actual file size : {__import__('os').path.getsize(OUTPUT_FILE) / 1e6:.1f} MB")

Generating 500,000 customer records...
  50,000 / 500,000 done...
  100,000 / 500,000 done...
  150,000 / 500,000 done...
  200,000 / 500,000 done...
  250,000 / 500,000 done...
  300,000 / 500,000 done...
  350,000 / 500,000 done...
  400,000 / 500,000 done...
  450,000 / 500,000 done...
  500,000 / 500,000 done...
Building DataFrame...

── Dataset Overview ──────────────────────
Shape            : (500000, 45)
File size (est.) : ~652.0 MB in memory

Churn distribution:
churn_type
Stayed           0.714
Company_Churn    0.167
Bundle_Switch    0.119

Overall churn rate : 16.69%
Avg ARPU           : ₹913.37
Avg tenure         : 29.1 months

Top bundles:
bundle_name
No_Bundle        108215
Entertainment     70151
Basic_OTT         70010
Premium_OTT       55415
Family_Bundle     45820

Saving to telecom_churn_data.csv...
Done. File saved: telecom_churn_data.csv
Actual file size : 109.5 MB


In [3]:
df.head(10)

,customer_id,region,age,gender,occupation,is_metro,plan_tier,contract_type,contract_length_months,tenure_months,...,peak_usage_hour,competitor_sim_owned,upgrade_offers_shown,upgrade_offers_accepted,retention_offers_6mo,retention_offer_accepted,loyalty_points_balance,churn_probability_score,churn_type,churn
0,CUST0000001,Hyderabad,21,Male,Salaried,1,Student_Pack,Quarterly,3,48,...,7,0,1,1,1,1,2531,0.2568,Stayed,0
1,CUST0000002,Mumbai,45,Male,Salaried,1,Prepaid_Premium,Monthly,1,57,...,7,0,0,0,2,1,5353,0.2578,Bundle_Switch,0
2,CUST0000003,Mumbai,25,Female,Homemaker,1,Prepaid_Basic,Monthly,1,3,...,6,1,1,0,0,0,1218,0.4350,Bundle_Switch,0
3,CUST0000004,Tier3_South,29,Male,Self_Employed,0,Postpaid_Family,2_Year,24,4,...,21,1,1,0,1,0,453,0.2788,Stayed,0
4,CUST0000005,Kolkata,31,Female,Homemaker,1,Prepaid_Premium,Quarterly,3,1,...,17,1,3,1,0,0,347,0.7455,Bundle_Switch,0
5,CUST0000006,Bangalore,38,Female,Salaried,1,Postpaid_Basic,2_Year,24,52,...,6,0,0,0,0,0,1752,0.0577,Stayed,0
6,CUST0000007,Jaipur,30,Male,Salaried,1,Postpaid_Family,Monthly,1,33,...,21,1,2,0,1,0,1258,0.3552,Stayed,0
7,CUST0000008,Mumbai,20,Male,Salaried,1,Prepaid_Basic,Annual,12,2,...,12,0,0,0,1,0,55,0.6359,Company_Churn,1
8,CUST0000009,Kolkata,22,Female,Salaried,1,Prepaid_Basic,Monthly,1,68,...,21,0,2,1,3,0,8264,0.3394,Stayed,0
9,CUST0000010,Tier3_North,30,Male,Self_Employed,0,Postpaid_Family,2_Year,24,13,...,22,0,0,0,1,0,3019,0.0437,Stayed,0


In [6]:
df.iloc[7]

customer_id                   CUST0000008
region                             Mumbai
age                                    20
gender                               Male
occupation                       Salaried
is_metro                                1
plan_tier                   Prepaid_Basic
contract_type                      Annual
contract_length_months                 12
tenure_months                           2
network_type                           5G
bundle_name                     Basic_OTT
ott_platforms                     Hotstar
num_ott_platforms                       1
monthly_bundle_cost                300.72
bundle_switches_6mo                     0
daily_data_limit_gb                   1.5
avg_daily_data_used_gb               1.17
data_usage_ratio                     0.78
total_data_used_gb_mo                35.1
data_overuse_days_mo                    0
monthly_call_minutes                  243
monthly_sms_count                      44
roaming_calls_mo                  

In [11]:
df['plan_tier'].value_counts()

plan_tier
Prepaid_Basic       114234
Prepaid_Mid         111328
Postpaid_Premium     75399
Prepaid_Premium      68866
Postpaid_Basic       62014
Postpaid_Family      52954
Student_Pack         15205
Name: count, dtype: int64